In [0]:
%run ./config

In [0]:
import boto3
import json
import uuid
import random
import time
from datetime import datetime,timedelta
from faker import Faker

In [0]:
fake=Faker()
s3=boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)

In [0]:
STATUSES=["placed","confirmed","shipped","delivered","cancelled"]
PRODUCTS=["Laptop","Phone","Headphones","Shoes","Watch","Backpack","Monitor","Keyboard"]


def generate_order():
    num_items=random.randint(1,4)
    items=[
        {
            "product":random.choice(PRODUCTS),
            "quantity":random.randint(1,3),
            "unit_price":round(random.uniform(10,500),2)
        }
        for _ in range(num_items)
    ]
    amount=round(sum(i['quantity']*i['unit_price'] for i in items),2)
    IST=datetime.utcnow()+timedelta(hours=5,minutes=30)
    return {
        "order_id":str(uuid.uuid4()),
        "customer_id":fake.uuid4(),
        "items":items,
        "amount":amount,
        "status":random.choice(STATUSES),
        "order_timestamp":IST.isoformat()
    }


In [0]:
def write_batch_to_s3(batch_size=25):
    now=datetime.utcnow()+timedelta(hours=5,minutes=30)
    orders=[generate_order() for _ in range(batch_size)]
    key=f"raw/orders/{now.strftime('%Y/%m/%d')}/order_{now.strftime('%H:%M:%S')}_{uuid.uuid4().hex[:8]}.json"
    body="\n".join(json.dumps(o) for o in orders)
    s3.put_object(Bucket=BUCKET_NAME,Key=key,Body=body)

    print(f"Wrote {batch_size} orders->s3://{BUCKET_NAME}/{key}")

In [0]:
NUM_BATCHES=10
BATCH_INTERVAL=15

for i in range(NUM_BATCHES):
    batch_size=random.randint(10,50)
    write_batch_to_s3(batch_size)

    if i<NUM_BATCHES-1:time.sleep(BATCH_INTERVAL)
print("Done Simulating Stream.!")